# v1.3 / 5 检测器中断恢复

原Notebook已完成3轮冻结骨干训练及25轮微调；从保留的最佳检查点继续2轮，并实际执行TFLite转换与验证。优化器重新建立，属于从权重续训，不宣称逐步完全复现未中断运行。

In [1]:
# v1.3: project-only Jupyter execution; no board programming.
import os,sys,json,hashlib
from pathlib import Path
import numpy as np
import tensorflow as tf
ROOT=Path.cwd()
while not (ROOT/'AGENTS.md').exists():
    if ROOT==ROOT.parent: raise RuntimeError('Open this notebook inside the CNN project')
    ROOT=ROOT.parent
assert ROOT.name=='fpga-w.-codex-cnn'
sys.path.insert(0,str(ROOT/'ml/tinyml'))
from workflow import export_int8,predict_tflite,digest
os.environ['KERAS_HOME']=str(ROOT/'ml/tinyml/keras')
tf.keras.utils.set_random_seed(1301)
tf.config.threading.set_intra_op_parallelism_threads(6)
tf.config.threading.set_inter_op_parallelism_threads(2)
print('TensorFlow',tf.__version__,'GPU',tf.config.list_physical_devices('GPU'))


TensorFlow 2.16.2 GPU []


In [2]:
sys.path.insert(0,str(ROOT/'ml/detector'))
from train import data,encode,augment,detector_loss
from export import stats
from workflow import split_detection_heads
training=data('train');validation=data('val')
out=ROOT/'ml/tinyml/artifacts/detector'
model=tf.keras.models.load_model(out/'best.keras',compile=False)
model.compile(tf.keras.optimizers.Adam(.0003),loss=detector_loss)


In [3]:
rng=np.random.default_rng(1301)
def generator():
    while True:
        for i in rng.permutation(len(training)):
            im,b,_=training[i];im,b=augment(im,b,rng)
            yield ((im.astype(np.float32)-128)/128)[...,None],encode(b)
options=tf.data.Options();options.threading.private_threadpool_size=4
ds=tf.data.Dataset.from_generator(generator,output_signature=(tf.TensorSpec((144,256,1),tf.float32),tf.TensorSpec((18,32,6),tf.float32))).batch(48).prefetch(2).with_options(options)
vx=np.stack([r[0] for r in validation])[...,None].astype(np.float32)
vy=np.stack([encode(r[1]) for r in validation])
vd=tf.data.Dataset.from_tensor_slices(((vx-128)/128,vy)).batch(48).with_options(options)
history=model.fit(ds,steps_per_epoch=(len(training)+47)//48,validation_data=vd,epochs=2,verbose=2,callbacks=[tf.keras.callbacks.ModelCheckpoint(str(out/'resumed_best.keras'),monitor='val_loss',save_best_only=True),tf.keras.callbacks.CSVLogger(str(out/'resumed_training.csv'))])
old_loss=float(model.evaluate(vd,verbose=0))
resumed=tf.keras.models.load_model(out/'resumed_best.keras',compile=False)
# Compare both saved candidates using validation only.
original=tf.keras.models.load_model(out/'best.keras',compile=False)
for candidate in (original,resumed):candidate.compile(loss=detector_loss)
a_loss=float(original.evaluate(vd,verbose=0));b_loss=float(resumed.evaluate(vd,verbose=0))
model=original if a_loss<=b_loss else resumed
model.save(out/'selected.keras')
print('Selected validation loss',min(a_loss,b_loss))


Epoch 1/2


C:\Users\francis\Desktop\fpga-w.-codex-cnn\ml\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


111/111 - 40s - 357ms/step - loss: 1.7332 - val_loss: 1.7442


Epoch 2/2


111/111 - 29s - 265ms/step - loss: 1.7189 - val_loss: 1.7335


Selected validation loss 1.7329394817352295


## 独立INT8检测头
分离object logit和坐标的输出量化尺度；两头浮点输出应与训练时的5通道输出完全一致。原先固定Q8的RTL不能直接执行这里的TFLite模型。

In [4]:
deployment=split_detection_heads(model)
p=model.predict((vx[:8]-128)/128,verbose=0)
a,b=deployment.predict((vx[:8]-128)/128,verbose=0)
assert np.max(np.abs(np.concatenate([a,b],axis=-1)-p))<1e-5
indices=np.random.default_rng(1301).choice(len(training),480,replace=False)
calibration=(np.stack([training[i][0] for i in indices])[...,None].astype(np.float32)-128)/128
path,contract=export_int8(deployment,calibration,out,'hand_detector_int8')
print(json.dumps(contract,indent=2))


C:\Users\francis\Desktop\fpga-w.-codex-cnn\ml\.venv\Lib\site-packages\tensorflow\lite\python\convert.py:964: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


{
  "version": "v1.3",
  "sha256": "98d072ef0fb673b3647fce234c559e92a3045e4fca09abcf281acccbbfaffc5f",
  "bytes": 48024,
  "inputs": [
    {
      "name": "gray",
      "shape": [
        1,
        144,
        256,
        1
      ],
      "dtype": "int8",
      "scale": 0.0078125,
      "zero_point": 0
    }
  ],
  "outputs": [
    {
      "name": "Identity",
      "shape": [
        1,
        18,
        32,
        1
      ],
      "dtype": "int8",
      "scale": 0.023937897756695747,
      "zero_point": 127
    },
    {
      "name": "Identity_1",
      "shape": [
        1,
        18,
        32,
        4
      ],
      "dtype": "int8",
      "scale": 0.004706344101577997,
      "zero_point": -77
    }
  ],
  "ops": [
    "CONV_2D",
    "DEPTHWISE_CONV_2D",
    "CONV_2D",
    "PAD",
    "DEPTHWISE_CONV_2D",
    "CONV_2D",
    "DEPTHWISE_CONV_2D",
    "CONV_2D",
    "PAD",
    "DEPTHWISE_CONV_2D",
    "CONV_2D",
    "DEPTHWISE_CONV_2D",
    "CONV_2D",
    "CONV_2D",
    "CONV_

In [5]:
outputs=predict_tflite(path,(vx-128)/128)
score=next(x for x in outputs if x.shape[-1]==1);boxes=next(x for x in outputs if x.shape[-1]==4)
q=np.floor(np.concatenate([score,boxes],axis=-1)*256+.5).astype(np.int32)
f=np.floor(model.predict((vx-128)/128,batch_size=48,verbose=0)*256+.5).astype(np.int32)
thresholds=[-768,-512,-384,-256,-128,0,128,256,512,768]
results=[stats(q,validation,t) for t in thresholds]
eligible=[r for r in results if r['precision_iou50']>=.85 and r['scene_recall_iou50']>=.50]
report=dict(version='v1.3',architecture='MobileNetV1 alpha .25 blocks1..5, gray conv1 folded, context conv32, separate INT8 heads',epochs=30,model_sha256=digest(out/'selected.keras'),tflite_sha256=digest(path),validation=results,float_validation=[stats(f,validation,t) for t in thresholds],deployment_gate_pass=bool(eligible),selected=max(eligible,key=lambda r:r['scene_recall_iou50']) if eligible else None,test_evaluated=False,hardware_verified=False)
(out/'validation_report.json').write_text(json.dumps(report,indent=2))
print(json.dumps(report,indent=2))


{
  "version": "v1.3",
  "architecture": "MobileNetV1 alpha .25 blocks1..5, gray conv1 folded, context conv32, separate INT8 heads",
  "epochs": 30,
  "model_sha256": "530ea5aee6c74eab6be90c5ad6220e38a46ce786bec8054ea2605f30d0ed12f2",
  "tflite_sha256": "98d072ef0fb673b3647fce234c559e92a3045e4fca09abcf281acccbbfaffc5f",
  "validation": [
    {
      "threshold_q8": -768,
      "images": 628,
      "positive_images": 432,
      "background_images": 196,
      "accepted": 602,
      "localized_iou50": 16,
      "precision_iou50": 0.026578073089700997,
      "scene_recall_iou50": 0.037037037037037035,
      "f1": 0.03094777562862669,
      "background_false_accept": 170,
      "mean_best_iou": 0.047380426257803095
    },
    {
      "threshold_q8": -512,
      "images": 628,
      "positive_images": 432,
      "background_images": 196,
      "accepted": 537,
      "localized_iou50": 16,
      "precision_iou50": 0.0297951582867784,
      "scene_recall_iou50": 0.037037037037037035,
      "f

## 下一步
门槛未通过：继续训练/诊断，保留实验TFLite用于算子兼容性检查，不发布部署结果。门槛通过：冻结阈值后再运行未使用的测试集和全流程定位+分类评估，再做目标TFLite Micro及硬件验证。